# Entity Identification Pipeline for Co-Pilot Agent

## Overview
This notebook implements a pipeline that predicts relevant entities from user queries in natural language. The goal is to identify which entities (like CDR, Phone, Web Actor, etc.) are mentioned or implied by the user's query.

## Approach
1. **Data Preprocessing**: Parse the training data and extract entity labels from JSON
2. **Feature Engineering**: Use entity field descriptions to enhance understanding
3. **Model Selection**: Use a small local LLM (DistilBERT for classification, with optional generative approach using TinyLlama/Phi)
4. **Evaluation**: Use multi-label classification metrics (accuracy, precision, recall, F1-score)

## 1. Import Dependencies

In [ ]:
# Install required packages if not already installed
# !pip install pandas numpy scikit-learn torch transformers accelerate sentencepiece

import pandas as pd
import numpy as np
import json
import ast
import re
from typing import List, Dict, Set, Tuple
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, hamming_loss, jaccard_score
)

# Deep learning imports
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, pipeline,
    AutoModelForCausalLM, BitsAndBytesConfig
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load and Explore Data

In [ ]:
# Load the datasets
user_queries_df = pd.read_csv('user_queries.csv')
fields_description_df = pd.read_csv('fields_description.csv')

print(f"User Queries Dataset Shape: {user_queries_df.shape}")
print(f"Fields Description Dataset Shape: {fields_description_df.shape}")
print("\n--- User Queries Sample ---")
display(user_queries_df.head())
print("\n--- Fields Description Sample ---")
display(fields_description_df.head(10))

In [ ]:
# Get unique entity types from fields description
entity_types = fields_description_df['entity_name'].unique()
print(f"Entity Types ({len(entity_types)}): {list(entity_types)}")

## 3. Extract Entity Labels from JSON

In [ ]:
def safe_parse_json(json_str: str) -> dict:
    """
    Safely parse a JSON string, handling Python dict format (with single quotes, True/False).
    """
    try:
        # Try standard JSON parsing first
        return json.loads(json_str)
    except json.JSONDecodeError:
        try:
            # Fall back to ast.literal_eval for Python dict format
            return ast.literal_eval(json_str)
        except (ValueError, SyntaxError) as e:
            print(f"Failed to parse: {json_str[:100]}... Error: {e}")
            return {}


def extract_entities_from_json(json_obj: dict) -> List[str]:
    """
    Extract all entities from a JSON object.
    Entities can be found in:
    - 'entityType' key (main entity)
    - 'relationTargetType' key (related entities, can be a list)
    """
    entities = set()
    
    # Extract main entityType
    if 'entityType' in json_obj:
        entities.add(json_obj['entityType'])
    
    # Recursively search for relationTargetType in statements
    def search_statements(statements):
        if not statements:
            return
        for statement in statements:
            if isinstance(statement, dict):
                # Check for relationTargetType in parameters
                params = statement.get('parameters', {})
                if 'relationTargetType' in params:
                    target_types = params['relationTargetType']
                    if isinstance(target_types, list):
                        entities.update(target_types)
                    else:
                        entities.add(target_types)
                
                # Recursively check nested statements
                if 'statements' in statement:
                    search_statements(statement['statements'])
    
    # Search in top-level statements
    if 'statements' in json_obj:
        search_statements(json_obj['statements'])
    
    return sorted(list(entities))


# Parse JSON and extract entities for each query
user_queries_df['parsed_json'] = user_queries_df['json'].apply(safe_parse_json)
user_queries_df['entities'] = user_queries_df['parsed_json'].apply(extract_entities_from_json)

# Display sample results
print("Sample extractions:")
for idx in [0, 7, 8, 14, 47]:
    if idx < len(user_queries_df):
        row = user_queries_df.iloc[idx]
        print(f"\nQuery: {row['question'][:80]}...")
        print(f"Entities: {row['entities']}")

In [ ]:
# Analyze entity distribution
all_entities = [entity for entities in user_queries_df['entities'] for entity in entities]
entity_counts = Counter(all_entities)

print("Entity Distribution:")
for entity, count in entity_counts.most_common():
    print(f"  {entity}: {count} ({100*count/len(user_queries_df):.1f}%)")

# Count queries with multiple entities
multi_entity_queries = sum(1 for entities in user_queries_df['entities'] if len(entities) > 1)
print(f"\nQueries with multiple entities: {multi_entity_queries} ({100*multi_entity_queries/len(user_queries_df):.1f}%)")

## 4. Build Entity Description Context

Use field descriptions to build a rich context for each entity type that can help the model understand what each entity represents.

In [ ]:
def build_entity_context(fields_df: pd.DataFrame) -> Dict[str, str]:
    """
    Build a descriptive context for each entity type based on its fields.
    """
    entity_context = {}
    
    for entity in fields_df['entity_name'].unique():
        entity_fields = fields_df[fields_df['entity_name'] == entity]
        
        # Create a summary of what this entity represents
        field_summaries = []
        for _, row in entity_fields.iterrows():
            field_name = row['field_name'].split('.')[-1]  # Get the last part of field name
            desc = row['description'][:100] if pd.notna(row['description']) else 'No description'
            field_summaries.append(f"{field_name}: {desc}")
        
        entity_context[entity] = {
            'field_count': len(entity_fields),
            'sample_fields': field_summaries[:5],
            'all_fields': field_summaries
        }
    
    return entity_context

entity_context = build_entity_context(fields_description_df)

# Print summary for each entity
print("Entity Contexts:")
for entity, context in entity_context.items():
    print(f"\n{entity} ({context['field_count']} fields):")
    for field in context['sample_fields'][:3]:
        print(f"  - {field[:80]}..." if len(field) > 80 else f"  - {field}")

## 5. Prepare Data for Training

In [ ]:
# Get all unique entity labels
all_unique_entities = sorted(list(set(all_entities)))
print(f"All unique entities ({len(all_unique_entities)}): {all_unique_entities}")

# Create label encoder for multi-label classification
mlb = MultiLabelBinarizer(classes=all_unique_entities)
y_encoded = mlb.fit_transform(user_queries_df['entities'])

print(f"\nEncoded labels shape: {y_encoded.shape}")
print(f"Label classes: {mlb.classes_}")

In [ ]:
# Split data into train, validation, and test sets
X = user_queries_df['question'].tolist()
y = y_encoded

# First split: 80% train+val, 20% test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Second split: 80% train, 20% val (of the trainval set)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

## 6. Approach 1: Fine-tuned DistilBERT Classifier

This approach uses a pre-trained DistilBERT model fine-tuned for multi-label classification. It's efficient and can run on CPU.

In [ ]:
class EntityDataset(Dataset):
    """
    PyTorch Dataset for entity classification.
    """
    def __init__(self, texts: List[str], labels: np.ndarray, tokenizer, max_length: int = 128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        labels = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(labels, dtype=torch.float)
        }

In [ ]:
# Initialize tokenizer and model
MODEL_NAME = "distilbert-base-uncased"
NUM_LABELS = len(all_unique_entities)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Number of labels: {NUM_LABELS}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Create datasets
train_dataset = EntityDataset(X_train, y_train, tokenizer)
val_dataset = EntityDataset(X_val, y_val, tokenizer)
test_dataset = EntityDataset(X_test, y_test, tokenizer)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

In [ ]:
def compute_metrics(eval_pred):
    """
    Compute evaluation metrics for multi-label classification.
    """
    predictions, labels = eval_pred
    predictions = (torch.sigmoid(torch.tensor(predictions)) > 0.5).numpy().astype(int)
    
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_micro': f1_score(labels, predictions, average='micro', zero_division=0),
        'f1_macro': f1_score(labels, predictions, average='macro', zero_division=0),
        'f1_weighted': f1_score(labels, predictions, average='weighted', zero_division=0),
        'precision_micro': precision_score(labels, predictions, average='micro', zero_division=0),
        'recall_micro': recall_score(labels, predictions, average='micro', zero_division=0),
        'hamming_loss': hamming_loss(labels, predictions),
        'jaccard_micro': jaccard_score(labels, predictions, average='micro', zero_division=0)
    }

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir='./entity_classifier_output',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_micro',
    greater_is_better=True,
    report_to='none',  # Disable wandb/tensorboard for local running
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
# Train the model
print("Starting training...")
trainer.train()
print("Training completed!")

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_results = trainer.evaluate(test_dataset)
print("\nTest Results:")
for metric, value in test_results.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Save the model
model.save_pretrained('./entity_classifier_model')
tokenizer.save_pretrained('./entity_classifier_model')
print("Model saved to ./entity_classifier_model")

## 7. Approach 2: Few-Shot Learning with Small LLM

This approach uses a small generative LLM with few-shot prompting. It's more flexible and doesn't require training.

In [ ]:
class FewShotEntityPredictor:
    """
    Entity predictor using few-shot learning with a small LLM.
    """
    
    def __init__(self, model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
        self.model_name = model_name
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        print(f"Loading model {model_name} on {self.device}...")
        
        # Load model with optimizations for CPU/low memory
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        if self.device == "cuda":
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16,
                device_map="auto"
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float32,
                low_cpu_mem_usage=True
            )
            self.model.to(self.device)
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        print("Model loaded successfully!")
    
    def build_prompt(self, query: str, examples: List[Tuple[str, List[str]]], 
                     entity_descriptions: Dict[str, str]) -> str:
        """
        Build a few-shot prompt for entity prediction.
        """
        # Entity type descriptions
        entity_desc = """
Available entity types:
- CDR: Call Detail Records - phone calls, SMS, emails, web communications
- Phone: Phone/device identifiers (IMEI, IMSI, MSISDN), phone metadata
- Web Activity: Social media posts, comments, online content from platforms
- Web Actor: Social media profiles/accounts on various platforms
- Person: Individual person records with personal information
- Investigation: Investigation records and cases
- Insight: Intelligence insights and analysis notes
- Report: Reports and documentation
- EVisa Request: Electronic visa application records
"""
        
        # Build few-shot examples
        examples_text = "\n".join([
            f"Query: {ex[0]}\nEntities: {ex[1]}"
            for ex in examples
        ])
        
        prompt = f"""You are an entity extraction system. Given a user query, identify which entity types are relevant.

{entity_desc}

Examples:
{examples_text}

Query: {query}
Entities:"""
        
        return prompt
    
    def predict(self, query: str, examples: List[Tuple[str, List[str]]], 
                entity_descriptions: Dict[str, str] = None) -> List[str]:
        """
        Predict entities for a given query using few-shot learning.
        """
        prompt = self.build_prompt(query, examples, entity_descriptions)
        
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=50,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract entities from response
        # Look for the part after "Entities:" in the response
        if "Entities:" in response:
            entities_text = response.split("Entities:")[-1].strip()
        else:
            entities_text = response
        
        # Parse entities from the response
        valid_entities = ['CDR', 'Phone', 'Web Activity', 'Web Actor', 'Person', 
                         'Investigation', 'Insight', 'Report', 'EVisa Request']
        
        predicted = []
        for entity in valid_entities:
            if entity.lower() in entities_text.lower():
                predicted.append(entity)
        
        return predicted if predicted else ['CDR']  # Default to CDR if nothing found

In [ ]:
# Create few-shot examples from training data
def get_few_shot_examples(df: pd.DataFrame, n_per_entity: int = 2) -> List[Tuple[str, List[str]]]:
    """
    Get balanced few-shot examples covering all entity types.
    """
    examples = []
    seen_entities = set()
    
    # First, get examples with multiple entities
    multi_entity = df[df['entities'].apply(len) > 1]
    for _, row in multi_entity.head(3).iterrows():
        examples.append((row['question'], row['entities']))
        seen_entities.update(row['entities'])
    
    # Then, get examples for each entity type
    for entity in df['entities'].explode().unique():
        if entity not in seen_entities:
            entity_rows = df[df['entities'].apply(lambda x: entity in x and len(x) == 1)]
            for _, row in entity_rows.head(n_per_entity).iterrows():
                examples.append((row['question'], row['entities']))
                seen_entities.add(entity)
    
    return examples[:15]  # Limit to 15 examples for context length

few_shot_examples = get_few_shot_examples(user_queries_df)
print(f"Few-shot examples ({len(few_shot_examples)}):")
for q, e in few_shot_examples[:5]:
    print(f"  Q: {q[:60]}... -> {e}")

In [ ]:
# Initialize the few-shot predictor (optional - requires model download)
# Uncomment to use:
# few_shot_predictor = FewShotEntityPredictor()

# Test prediction
# test_query = "What SMS messages were sent from suspicious phones?"
# predicted = few_shot_predictor.predict(test_query, few_shot_examples)
# print(f"Query: {test_query}")
# print(f"Predicted entities: {predicted}")

## 8. Approach 3: Rule-Based + Embedding Similarity (Lightweight Alternative)

This approach combines rule-based keyword matching with embedding similarity. It's lightweight and fast, suitable for CPU-only environments.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

class HybridEntityPredictor:
    """
    Hybrid entity predictor combining:
    1. Rule-based keyword matching
    2. Embedding similarity search
    """
    
    def __init__(self, embedding_model: str = "all-MiniLM-L6-v2"):
        # Load sentence transformer for embeddings
        self.embedding_model = SentenceTransformer(embedding_model)
        
        # Define entity keywords for rule-based matching
        self.entity_keywords = {
            'CDR': ['call', 'sms', 'text message', 'email', 'communication', 'phone call', 
                   'voice', 'msisdn', 'imei', 'imsi', 'duration', 'caller', 'callee',
                   'sent', 'received', 'blocked', 'failed', 'redirected', 'web communication'],
            'Phone': ['phone', 'device', 'mobile', 'suspicious phone', 'target phone',
                     'landline', 'foreign phone', 'phone number', 'sim', 'participant'],
            'Web Activity': ['post', 'comment', 'tweet', 'activity', 'social media',
                            'facebook', 'twitter', 'instagram', 'reddit', 'youtube',
                            'linkedin', 'like', 'share', 'hashtag', 'mention', 'sentiment'],
            'Web Actor': ['profile', 'account', 'user profile', 'channel', 'page',
                         'follower', 'following', 'friends', 'web actor', 'social profile'],
            'Person': ['person', 'individual', 'people', 'man', 'woman', 'male', 'female',
                      'first name', 'last name', 'birth date', 'passport', 'occupation'],
            'Investigation': ['investigation', 'case', 'inquiry', 'probe', 'open case',
                             'closed case', 'priority', 'due date'],
            'Insight': ['insight', 'intelligence', 'analysis', 'finding', 'assessment'],
            'Report': ['report', 'document', 'documentation', 'summary', 'briefing'],
            'EVisa Request': ['visa', 'evisa', 'travel', 'arrival', 'departure', 
                             'visitor', 'citizenship', 'passport', 'immigration']
        }
        
        self.nn_model = None
        self.train_entities = None
        self.train_embeddings = None
    
    def fit(self, queries: List[str], entities: List[List[str]]):
        """
        Fit the embedding-based similarity search.
        """
        print("Computing embeddings for training queries...")
        self.train_embeddings = self.embedding_model.encode(queries, show_progress_bar=True)
        self.train_entities = entities
        
        # Fit nearest neighbors
        self.nn_model = NearestNeighbors(n_neighbors=5, metric='cosine')
        self.nn_model.fit(self.train_embeddings)
        print("Model fitted!")
    
    def predict_rule_based(self, query: str) -> Set[str]:
        """
        Predict entities using keyword matching.
        """
        query_lower = query.lower()
        predicted = set()
        
        for entity, keywords in self.entity_keywords.items():
            for keyword in keywords:
                if keyword in query_lower:
                    predicted.add(entity)
                    break
        
        return predicted
    
    def predict_embedding(self, query: str, k: int = 5) -> List[str]:
        """
        Predict entities using embedding similarity.
        """
        if self.nn_model is None:
            return []
        
        query_embedding = self.embedding_model.encode([query])
        distances, indices = self.nn_model.kneighbors(query_embedding)
        
        # Aggregate entities from nearest neighbors with voting
        entity_votes = Counter()
        for idx, dist in zip(indices[0], distances[0]):
            weight = 1 - dist  # Higher weight for closer neighbors
            for entity in self.train_entities[idx]:
                entity_votes[entity] += weight
        
        # Return entities with significant votes
        if entity_votes:
            max_vote = max(entity_votes.values())
            return [e for e, v in entity_votes.items() if v >= max_vote * 0.5]
        return []
    
    def predict(self, query: str, use_rules: bool = True, use_embedding: bool = True) -> List[str]:
        """
        Predict entities using hybrid approach.
        """
        predicted = set()
        
        if use_rules:
            predicted.update(self.predict_rule_based(query))
        
        if use_embedding:
            predicted.update(self.predict_embedding(query))
        
        return sorted(list(predicted)) if predicted else ['CDR']

In [ ]:
# Initialize and train hybrid predictor
hybrid_predictor = HybridEntityPredictor()

# Fit on training data
train_queries = [user_queries_df.iloc[i]['question'] for i in range(len(X_train))]
train_entities = [user_queries_df.iloc[i]['entities'] for i in range(len(X_train))]

# Actually use the split data
hybrid_predictor.fit(X_train, [mlb.inverse_transform(y_train[i:i+1])[0] for i in range(len(y_train))])

## 9. Unified Prediction Pipeline

In [ ]:
class EntityIdentificationPipeline:
    """
    Unified pipeline for entity identification supporting multiple approaches.
    """
    
    def __init__(self, approach: str = 'classifier'):
        """
        Initialize the pipeline.
        
        Args:
            approach: One of 'classifier', 'few_shot', 'hybrid'
        """
        self.approach = approach
        self.model = None
        self.tokenizer = None
        self.mlb = None
        self.hybrid_predictor = None
        self.few_shot_examples = None
        
    def load_classifier(self, model_path: str = './entity_classifier_model'):
        """
        Load the fine-tuned classifier model.
        """
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_path)
        self.model.eval()
        
        # Load label encoder
        self.mlb = MultiLabelBinarizer(classes=[
            'CDR', 'EVisa Request', 'Insight', 'Investigation', 
            'Person', 'Phone', 'Report', 'Web Activity', 'Web Actor'
        ])
        self.mlb.fit([[]])  # Initialize with empty
        
        print(f"Classifier loaded from {model_path}")
    
    def load_hybrid(self, train_queries: List[str], train_entities: List[List[str]]):
        """
        Initialize the hybrid predictor.
        """
        self.hybrid_predictor = HybridEntityPredictor()
        self.hybrid_predictor.fit(train_queries, train_entities)
        print("Hybrid predictor initialized")
    
    def set_few_shot_examples(self, examples: List[Tuple[str, List[str]]]):
        """
        Set few-shot examples for the few-shot approach.
        """
        self.few_shot_examples = examples
    
    def predict(self, query: str, threshold: float = 0.5) -> List[str]:
        """
        Predict entities for a given query.
        """
        if self.approach == 'classifier':
            return self._predict_classifier(query, threshold)
        elif self.approach == 'hybrid':
            return self._predict_hybrid(query)
        else:
            raise ValueError(f"Unknown approach: {self.approach}")
    
    def _predict_classifier(self, query: str, threshold: float = 0.5) -> List[str]:
        """
        Predict using the fine-tuned classifier.
        """
        if self.model is None:
            raise ValueError("Classifier not loaded. Call load_classifier() first.")
        
        inputs = self.tokenizer(
            query, 
            return_tensors='pt', 
            truncation=True, 
            padding=True,
            max_length=128
        )
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.sigmoid(outputs.logits)
        
        predictions = (probs > threshold).numpy().astype(int)
        entities = self.mlb.inverse_transform(predictions)
        
        return list(entities[0]) if entities[0] else ['CDR']
    
    def _predict_hybrid(self, query: str) -> List[str]:
        """
        Predict using the hybrid approach.
        """
        if self.hybrid_predictor is None:
            raise ValueError("Hybrid predictor not initialized. Call load_hybrid() first.")
        
        return self.hybrid_predictor.predict(query)
    
    def predict_with_confidence(self, query: str) -> Dict[str, float]:
        """
        Predict with confidence scores (classifier only).
        """
        if self.approach != 'classifier' or self.model is None:
            raise ValueError("Confidence scores only available for classifier approach")
        
        inputs = self.tokenizer(
            query,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=128
        )
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.sigmoid(outputs.logits).numpy()[0]
        
        return {entity: float(prob) for entity, prob in zip(self.mlb.classes_, probs)}

## 10. Evaluation Functions

In [ ]:
def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray, 
                        label_names: List[str]) -> Dict[str, float]:
    """
    Comprehensive evaluation of multi-label predictions.
    """
    metrics = {
        'exact_match_accuracy': accuracy_score(y_true, y_pred),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'jaccard_micro': jaccard_score(y_true, y_pred, average='micro', zero_division=0),
        'jaccard_macro': jaccard_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'precision_micro': precision_score(y_true, y_pred, average='micro', zero_division=0),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_micro': recall_score(y_true, y_pred, average='micro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
    }
    
    print("=" * 60)
    print("EVALUATION METRICS")
    print("=" * 60)
    print(f"\nOverall Metrics:")
    print(f"  Exact Match Accuracy: {metrics['exact_match_accuracy']:.4f}")
    print(f"  Hamming Loss: {metrics['hamming_loss']:.4f}")
    print(f"  Jaccard Score (micro): {metrics['jaccard_micro']:.4f}")
    print(f"\nF1 Scores:")
    print(f"  F1 Micro: {metrics['f1_micro']:.4f}")
    print(f"  F1 Macro: {metrics['f1_macro']:.4f}")
    print(f"  F1 Weighted: {metrics['f1_weighted']:.4f}")
    print(f"\nPrecision/Recall:")
    print(f"  Precision (micro): {metrics['precision_micro']:.4f}")
    print(f"  Recall (micro): {metrics['recall_micro']:.4f}")
    
    # Per-class metrics
    print(f"\nPer-Class Report:")
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
    
    return metrics


def run_test_cases(pipeline, test_queries: List[str], expected_entities: List[List[str]]):
    """
    Run test cases and display results.
    """
    print("\n" + "=" * 60)
    print("TEST CASES")
    print("=" * 60)
    
    correct = 0
    partial_correct = 0
    
    for i, (query, expected) in enumerate(zip(test_queries, expected_entities)):
        predicted = pipeline.predict(query)
        
        # Check correctness
        expected_set = set(expected)
        predicted_set = set(predicted)
        
        is_exact = expected_set == predicted_set
        is_partial = len(expected_set & predicted_set) > 0
        
        if is_exact:
            correct += 1
            status = "✓ EXACT"
        elif is_partial:
            partial_correct += 1
            status = "~ PARTIAL"
        else:
            status = "✗ WRONG"
        
        print(f"\nTest {i+1}: {status}")
        print(f"  Query: {query[:70]}..." if len(query) > 70 else f"  Query: {query}")
        print(f"  Expected: {sorted(expected)}")
        print(f"  Predicted: {sorted(predicted)}")
    
    print(f"\n" + "=" * 60)
    print(f"Summary: {correct}/{len(test_queries)} exact matches, "
          f"{partial_correct}/{len(test_queries)} partial matches")
    print(f"Exact Match Rate: {100*correct/len(test_queries):.1f}%")

## 11. Run Evaluation on Hybrid Model

In [ ]:
# Evaluate hybrid predictor on test set
print("Evaluating Hybrid Predictor on Test Set...")

y_pred_hybrid = []
for query in X_test:
    pred = hybrid_predictor.predict(query)
    y_pred_hybrid.append(pred)

# Convert predictions to binary format
y_pred_hybrid_encoded = mlb.transform(y_pred_hybrid)

# Evaluate
hybrid_metrics = evaluate_predictions(y_test, y_pred_hybrid_encoded, mlb.classes_)

## 12. Test Cases

In [ ]:
# Define test cases
test_cases = [
    {
        "query": "What SMS messages were sent from suspicious phones to 0549876543 containing the word 'urgent'?",
        "expected": ["CDR", "Phone"]
    },
    {
        "query": "Find all calls made using 3G technology",
        "expected": ["CDR"]
    },
    {
        "query": "Show me all tweets from accounts with 500 friends mentioning Tesla",
        "expected": ["Web Activity", "Web Actor"]
    },
    {
        "query": "Which phones have been marked as suspicious?",
        "expected": ["Phone"]
    },
    {
        "query": "Find all individuals with the occupation 'engineer' born before July 1985",
        "expected": ["Person"]
    },
    {
        "query": "Show me investigations that are open or were created in the last 3 months",
        "expected": ["Investigation"]
    },
    {
        "query": "Find all insights containing 'money laundering' created in the past month",
        "expected": ["Insight"]
    },
    {
        "query": "List all visitors whose travel document was issued before January 2020",
        "expected": ["EVisa Request"]
    },
    {
        "query": "Get reports that were created in the past 3 days",
        "expected": ["Report"]
    },
    {
        "query": "Find Instagram profiles with 100 followers that use Israel phone number",
        "expected": ["Web Actor"]
    },
    {
        "query": "Show me all Reddit posts that mention a phone number and an email address",
        "expected": ["Web Activity"]
    },
    {
        "query": "List emails sent to phones associated with the target Sarah Johnson",
        "expected": ["CDR", "Phone"]
    }
]

test_queries = [tc["query"] for tc in test_cases]
test_expected = [tc["expected"] for tc in test_cases]

# Run test cases with hybrid predictor
print("\nHybrid Predictor Results:")
run_test_cases_simple(hybrid_predictor, test_queries, test_expected)

def run_test_cases_simple(predictor, queries, expected):
    """Simple test case runner for predictors."""
    correct = 0
    partial = 0
    
    for i, (q, exp) in enumerate(zip(queries, expected)):
        pred = predictor.predict(q)
        exp_set = set(exp)
        pred_set = set(pred)
        
        if exp_set == pred_set:
            correct += 1
            status = "✓"
        elif exp_set & pred_set:
            partial += 1
            status = "~"
        else:
            status = "✗"
        
        print(f"{status} Test {i+1}: {q[:50]}...")
        print(f"   Expected: {sorted(exp)}, Got: {sorted(pred)}")
    
    print(f"\nResults: {correct}/{len(queries)} exact, {partial}/{len(queries)} partial")

In [ ]:
# Run test cases
run_test_cases_simple(hybrid_predictor, test_queries, test_expected)

## 13. Summary and Recommendations

### Approaches Implemented

1. **Fine-tuned DistilBERT Classifier** (Recommended for production)
   - Pros: High accuracy, fast inference, well-suited for multi-label classification
   - Cons: Requires training, needs GPU for efficient training
   - Best for: Production deployment with consistent entity types

2. **Few-Shot Learning with Small LLM**
   - Pros: No training required, flexible, can handle new entity types
   - Cons: Slower inference, requires careful prompt engineering
   - Best for: Prototyping, handling edge cases

3. **Hybrid Rule-Based + Embedding**
   - Pros: Fast, interpretable, no GPU required
   - Cons: Requires manual keyword maintenance, may miss nuanced queries
   - Best for: CPU-only environments, quick deployment

### Metrics Used

- **Exact Match Accuracy**: Percentage of queries where all predicted entities exactly match ground truth
- **F1 Score (Micro/Macro)**: Harmonic mean of precision and recall
- **Hamming Loss**: Fraction of incorrectly predicted labels
- **Jaccard Score**: Intersection over union of predicted and true labels

### Open Issues and Future Improvements

1. **Data Augmentation**: Generate synthetic queries to improve model robustness
2. **Active Learning**: Implement feedback loop to improve on edge cases
3. **Ensemble Methods**: Combine multiple approaches for better accuracy
4. **Context Enhancement**: Use entity field descriptions more effectively
5. **Threshold Tuning**: Optimize classification threshold per entity type
6. **Error Analysis**: Detailed analysis of failure cases to guide improvements

In [ ]:
# Save configuration and results
import json

config = {
    "entity_types": all_unique_entities,
    "dataset_size": len(user_queries_df),
    "train_size": len(X_train),
    "val_size": len(X_val),
    "test_size": len(X_test),
    "approaches": ["classifier", "few_shot", "hybrid"],
    "recommended_approach": "classifier"
}

with open('pipeline_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Configuration saved to pipeline_config.json")